In [ ]:
import torch
import pickle
import numpy as np
import torch.nn as nn
import torch.optim as optim

from tqdm import tqdm
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms

## Task I: Word-based CNN for Text Classification

### 1. Data

The dataset that we are going to use is the imdb dataset of movie reviews. These are labelled by sentiment (positive/negative).

The reviews have been preprocessed, and each review is encoded as a sequence of word indexes (integers).

For convenience, words are indexed by overall frequency in the dataset, so that for instance the integer "3" encodes the 3rd most frequent word in the data. This allows for quick filtering operations such as: "only consider the top 10,000 most common words, but eliminate the top 20 most common words".

More information regarding the dataset can be found in the official [documentation](https://keras.io/datasets/#imdb-movie-reviews-sentiment-classification).


In [ ]:
# Load the data
train_samples, train_labels, test_samples, test_labels = pickle.load(open('./imdb.pkl', 'rb'))

print(len(train_samples), 'train sequences')
print(len(test_samples), 'test sequences')

25000 train sequences
25000 test sequences


### 2. Preprocess the text data

In this particular case, where we are using the imdb dataset there is no need to do all the traditional preprocessings that we normally do when dealing with NLP problems. Part of them are already done at this point.

  - Split the dataset in train and test (maybe also validation).
  - Tokenize and transform to integer index. Here we would need to:
    - instantitate a *Tokenizer()* object,
    - fit that object on the text on which we are training the model (use the *fit_on_texts()* method)
    - call *texts_to_sequences()* for both the training and the test text.

  - **Add padding to ensure that all vectors have the same dimensionality.** Note that this is the only pre-processing that needs to be done in the case of the current imdb dataset.

In [ ]:
class ToNumpy:
    def __call__(self, sample):
        return np.array(sample)


class PadAndTruncate:
    def __init__(self, max_len, pad_value):
        assert isinstance(max_len, int) and max_len >= 0, "max_len must be a positive integer"
        assert isinstance(pad_value, int), "pad_value must be an integer"

        self.max_len = max_len
        self.pad_value = pad_value

    def __call__(self, sample):
        seq_len = sample.shape[0]

        if seq_len > self.max_len:
            return sample[:self.max_len]
        if seq_len < self.max_len:
            return np.concatenate([sample, [self.pad_value] * (self.max_len - seq_len)])
        return sample


class IMDBDataset(Dataset):
    def __init__(self, samples, labels, transform=None):
        self.samples = samples
        self.labels = labels
        self.transform = transform

    def __getitem__(self, id):
        sample = self.samples[id]
        label = self.labels[id]

        if self.transform:
            sample = self.transform(sample)

        return sample, label

    def __len__(self):
        return self.samples.shape[0]


In [ ]:
class Model(nn.Module):
    def __init__(self, vocab_size=10002, embedding_dim=100, padding_idx=10001):
        super().__init__()

        # Define an embedding layer with a vocabulary size of 10002
        # an output embedding size of 100
        # and a padding_idx equal to the one used - 10001
        self.embedding = nn.Embedding(
            num_embeddings=vocab_size,
            embedding_dim=embedding_dim,
            padding_idx=padding_idx
        )

        # Define the following sequence of layers

        # A dropout layer with a probability of 0.4
        self.dropout = nn.Dropout(0.4)

        self.conv_blocks = nn.Sequential(
            # A 1D Convolutional layer with 100 input channels, 128 output channels, kernel size of 3 and a padding of 1
            nn.Conv1d(in_channels=embedding_dim, out_channels=128, kernel_size=3, padding=1),
            # A 1D Batch Normalization Layer for 128 features
            nn.BatchNorm1d(num_features=128),
            # A ReLU activation
            nn.ReLU(),
            # A 1D Maxpooling layer with size 2
            nn.MaxPool1d(kernel_size=2),

            # A 1D Convolutional layer with 128 input channels, 128 output channels, kernel size of 5 and a padding of 2
            nn.Conv1d(in_channels=128, out_channels=128, kernel_size=5, padding=2),
            # A 1D Batch Normalization Layer for 128 features
            nn.BatchNorm1d(num_features=128),
            # A ReLU activation
            nn.ReLU(),
            # A 1D Maxpooling layer with size 2
            nn.MaxPool1d(kernel_size=2),

            # A 1D Convolutional layer with 128 input channels, 128 output channels, kernel size of 5 and a padding of 2
            nn.Conv1d(in_channels=128, out_channels=128, kernel_size=5, padding=2),
            # A 1D Batch Normalization Layer for 128 features
            nn.BatchNorm1d(num_features=128),
            # A ReLU activation
            nn.ReLU(),
            # A 1D Maxpooling layer with size 2
            nn.MaxPool1d(kernel_size=2))

        # A global Average pooling layer, which in this scenario, will be an 1D Avgerage Pooling layer
        # with size 125 and stride 125
        self.global_avg_pool = nn.AvgPool1d(kernel_size=125, stride=125)

        # A Flattening layer
        self.flatten = nn.Flatten()

        # A Linear layer with 128 input features and 2 outputs and no activation function
        self.fc = nn.Linear(in_features=128, out_features=2)


    def forward(self, input_ids):
        # forward the input through the embedding layer
        x = self.embedding(input_ids)

        # permute the input such that it becomes channels first
        x = x.permute(0, 2, 1)

        # forward the input through the rest of the sequence of layers
        x = self.dropout(x)
        x = self.conv_blocks(x)
        x = self.global_avg_pool(x)
        x = self.flatten(x)
        output = self.fc(x)

        return output

In [ ]:
class Model(nn.Module):
    def __init__(self, vocab_size=10002, embedding_dim=100, padding_idx=10001):
        self.first_fwd = False
        super().__init__()

        self.embedding = nn.Embedding(
            num_embeddings=vocab_size,
            embedding_dim=embedding_dim,
            padding_idx=padding_idx
        )

        self.dropout = nn.Dropout(0.4)

        self.conv_parallel_3 = nn.Sequential(
            nn.Conv1d(in_channels=embedding_dim, out_channels=128, kernel_size=3, padding=1),
            nn.BatchNorm1d(num_features=128),
            nn.ReLU(),
        )
        self.conv_parallel_5 = nn.Sequential(
            nn.Conv1d(in_channels=embedding_dim, out_channels=128, kernel_size=5, padding=2),
            nn.BatchNorm1d(num_features=128),
            nn.ReLU(),
        )

        self.max_pool_parallel = nn.MaxPool1d(kernel_size=2)

        self.conv_blocks_rest = nn.Sequential(
            nn.Conv1d(in_channels=256, out_channels=128, kernel_size=5, padding=2),
            nn.BatchNorm1d(num_features=128),
            nn.ReLU(),
            nn.MaxPool1d(kernel_size=2),

            nn.Conv1d(in_channels=128, out_channels=128, kernel_size=5, padding=2),
            nn.BatchNorm1d(num_features=128),
            nn.ReLU(),
            nn.MaxPool1d(kernel_size=2)
        )


        self.global_avg_pool = nn.AvgPool1d(kernel_size=125, stride=125)

        self.flatten = nn.Flatten()

        self.fc = nn.Linear(in_features=128, out_features=2)


    def forward(self, input_ids):
        x = self.embedding(input_ids)
        if not self.first_fwd:
            print(f"After embedding shape: {x.shape}")
        x = x.permute(0, 2, 1)
        x = self.dropout(x)
        if not self.first_fwd:
            print(f"After dropout shape: {x.shape}")

        x_k3 = self.conv_parallel_3(x)
        x_k5 = self.conv_parallel_5(x)
        if not self.first_fwd:
            print(f"x_k3 shape: {x_k3.shape}")
            print(f"x_k5 shape: {x_k5.shape}")

        x = torch.cat((x_k3, x_k5), dim=1)
        if not self.first_fwd:
            print(f"After concat shape: {x.shape}")

        x = self.max_pool_parallel(x)
        if not self.first_fwd:
            print(f"After max pool shape: {x.shape}")

        x = self.conv_blocks_rest(x)
        if not self.first_fwd:
            print(f"After conv blocks shape: {x.shape}")
        x = self.global_avg_pool(x)
        if not self.first_fwd:
            print(f"After global avg pool shape: {x.shape}")
        x = self.flatten(x)
        if not self.first_fwd:
            print(f"After flatten shape: {x.shape}")
        output = self.fc(x)

        self.first_fwd = True
        return output

### 3.  Define the model de dataset and the training loop

Similar to the privious lab while following the model architecture described in the comments.

In [ ]:
desired_length = 1000
pad_value = 10001
batch_size = 64

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}\n")

# instantiate the model
model = Model(
    vocab_size=10002,
    embedding_dim=100,
    padding_idx=pad_value
).to(device)

# define an Adam optimizer for the model with a lr of 1e-3
optimizer = optim.Adam(model.parameters(), lr=1e-3)

# define a Cross Entropy loss function
criterion = nn.CrossEntropyLoss()

# define the training dataset and dataloader and the test dataset and dataloader
imdb_ds_transforms = transforms.Compose([
    ToNumpy(),
    PadAndTruncate(desired_length, pad_value)
])

train_ds = IMDBDataset(samples=train_samples, labels=train_labels, transform=imdb_ds_transforms)
test_ds = IMDBDataset(samples=test_samples, labels=test_labels, transform=imdb_ds_transforms)

train_dl = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
test_dl = DataLoader(test_ds, batch_size=batch_size, shuffle=False)


# write the training loop as defined in Lab 1 and train the model
num_epochs = 1
print(f"Starting training for {num_epochs} epochs...")

for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    correct_preds = 0
    total_samples = 0

    train_pbar = tqdm(train_dl, desc=f"Epoch {epoch+1}/{num_epochs} [Train]")

    for inputs, labels in train_pbar:
        inputs = inputs.to(device, dtype=torch.long)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(inputs)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        running_loss += loss.item() * inputs.size(0)
        _, preds = torch.max(outputs, 1)
        correct_preds += torch.sum(preds == labels.data)
        total_samples += labels.size(0)

        train_pbar.set_postfix({
            'loss': running_loss / total_samples,
            'acc': (correct_preds.double() / total_samples).item()
        })

    epoch_train_loss = running_loss / total_samples
    epoch_train_acc = (correct_preds.double() / total_samples).item()

    model.eval()
    running_loss = 0.0
    correct_preds = 0
    total_samples = 0

    test_pbar = tqdm(test_dl, desc=f"Epoch {epoch+1}/{num_epochs} [Test]")

    with torch.no_grad():
        for inputs, labels in test_pbar:
            inputs = inputs.to(device, dtype=torch.long)
            labels = labels.to(device)

            outputs = model(inputs)
            loss = criterion(outputs, labels)

            running_loss += loss.item() * inputs.size(0)
            _, preds = torch.max(outputs, 1)
            correct_preds += torch.sum(preds == labels.data)
            total_samples += labels.size(0)

            test_pbar.set_postfix({
                'loss': running_loss / total_samples,
                'acc': (correct_preds.double() / total_samples).item()
            })

    epoch_test_loss = running_loss / total_samples
    epoch_test_acc = (correct_preds.double() / total_samples).item()

    print(f"Epoch {epoch+1}/{num_epochs} | "
          f"Train Loss: {epoch_train_loss:.4f} Acc: {epoch_train_acc:.4f} | "
          f"Test Loss: {epoch_test_loss:.4f} Acc: {epoch_test_acc:.4f}")

print("\nTraining finished.")

Using device: cuda

Starting training for 1 epochs...


Epoch 1/1 [Train]:   1%|          | 2/391 [00:00<00:25, 15.27it/s, loss=0.692, acc=0.469]

After embedding shape: torch.Size([64, 1000, 100])
After dropout shape: torch.Size([64, 100, 1000])
x_k3 shape: torch.Size([64, 128, 1000])
x_k5 shape: torch.Size([64, 128, 1000])
After concat shape: torch.Size([64, 256, 1000])
After max pool shape: torch.Size([64, 256, 500])
After conv blocks shape: torch.Size([64, 128, 125])
After global avg pool shape: torch.Size([64, 128, 1])
After flatten shape: torch.Size([64, 128])


Epoch 1/1 [Test]: 100%|██████████| 391/391 [00:06<00:00, 58.11it/s, loss=0.419, acc=0.825]

Epoch 1/1 | Train Loss: 0.5325 Acc: 0.7343 | Test Loss: 0.4191 Acc: 0.8254

Training finished.
